In [ ]:
import pandas as pd
from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd()))
from sinan_mappings import add_sinan_cbo_labels, standardize_columns


df17 = pd.read_parquet('../data/DENGBR17.parquet')
df18 = pd.read_parquet('../data/DENGBR18.parquet')
df19 = pd.read_parquet('../data/DENGBR19.parquet')

df = pd.concat([df17, df18, df19], ignore_index=True)
df = standardize_columns(df)

colunas_pedro = [
    'residence_health_region', 'complications', 'notification_date', 'gum_bleeding', 'residence_country', 'severity_signs_date',
    'alarm_bleeding', 'chik_test2_date', 'chik_test2_result', 'other_bleeding', 'work_related', 'disease_code',
    'alarm_persistent_vomit', 'serology_date', 'severe_cold_extremities', 'fever', 'notification_year',
    'chik_test1_date', 'platelets_below_100k', 'sex', 'alarm_hypotension', 'autochthonous_case', 'hospitalized',
    'health_facility', 'occupation_code', 'case_outcome', 'severe_melena', 'nausea', 'viral_isolation_date',
    'age', 'alarm_lethargy', 'diabetes', 'return_flow_flag', 'hospital_municipality', 'residence_state',
    'confirmation_criteria', 'notif_health_region'
]

colunas_target = [
    'residence_health_region', 'notification_date', 'disease_code',
    'fever', 'notification_year', 'sex', 'autochthonous_case', 'hospitalized',
    'health_facility', 'occupation_code', 'nausea',
    'notif_health_region', 'residence_state', 'age'
]

df_filtrado = df[colunas_target].copy()

def parse_idade(valor):
    try:
        s = str(int(valor)).zfill(4)
        unidade = int(s[0])
        numero  = int(s[1:])
        if unidade == 4:
            return numero
        elif unidade == 3:
            return numero / 12
        elif unidade == 2:
            return numero / 365
        elif unidade == 1:
            return numero / 8760
        else:
            return None
    except:
        return None

df_filtrado['age_years'] = df_filtrado['age'].apply(parse_idade)
df_filtrado = df_filtrado.drop(columns=['age'])


df_filtrado['notification_date'] = pd.to_datetime(df_filtrado['notification_date'], format='%Y-%m-%d')
df_filtrado['notification_month'] = df_filtrado['notification_date'].dt.month
df_filtrado['notification_day'] = df_filtrado['notification_date'].dt.day
df_filtrado = df_filtrado.drop(columns=['notification_date'])

df_filtrado['occupation_code'] = df_filtrado['occupation_code'].fillna(0)
df_filtrado = df_filtrado.drop(columns=['autochthonous_case'])

df_filtrado = add_sinan_cbo_labels(df_filtrado)

print(list(df_filtrado))
display(df_filtrado)
#print(df_filtrado.shape)
#print(df_filtrado.isnull().sum())